# CLM-8B on Reuters, the G-SIB benchmarks, FinCausal and full Bloomberg
**What you do:** check the GPU is **L4** (*Runtime → Change runtime type*), then **Runtime → Run all**, and approve Google Drive when asked.

**This takes days of GPU time, over several sessions.** Progress is saved to Google Drive (`MyDrive/fin-decider-clm/`). Whenever Colab stops (the 24-hour limit, a closed tab, a disconnect), just open this notebook again and **Run all**: it skips finished work and continues.

Each finished dataset is also downloaded to your computer as `clm_<dataset>.zip` (allow multiple downloads if the browser asks). Optional: to include the gated credit-agreements dataset, add your Hugging Face token under 🔑 *Secrets* (left sidebar) as `HF_TOKEN` with notebook access on; otherwise it's skipped.

In [ ]:
# Step 1: GPU check (needs ~24 GB: L4 or A100) and Google Drive for saved progress.
import subprocess, os
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
assert gpu and int(gpu.split(',')[-1]) > 20000, 'GPU too small: Runtime -> Change runtime type -> L4, then Runtime -> Run all.'
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/fin-decider-clm'
os.makedirs(SAVE, exist_ok=True)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN found: credit agreements will run')
except Exception:
    print('No HF_TOKEN secret: credit agreements will be skipped')

In [ ]:
# Step 2: code, packages, and links so corpora and results live on Drive (about 5-10 minutes).
%cd /content
!rm -rf fin-decider && git clone -q -b isaac_cwb https://github.com/bansalr/fin-decider.git
%cd /content/fin-decider
!git log --oneline -1
!pip install -q vllm==0.32.0 contrastive-lm==0.1.0 2>&1 | tail -2
!pip install -q -e . 2>&1 | tail -2
for d in ['corpus', 'corpus_bench', 'corpus_fincausal', 'corpus_native', 'classification_results', 'classification_results_native', 'reports', 'logs']:
    os.makedirs(f'{SAVE}/{d}', exist_ok=True)
    if not os.path.islink(d):
        subprocess.run(['rm', '-rf', d]); os.symlink(f'{SAVE}/{d}', d)
!pip freeze > {SAVE}/logs/pip_freeze.txt; nvidia-smi > {SAVE}/logs/gpu.txt; git rev-parse HEAD > {SAVE}/logs/commit.txt

In [ ]:
# Step 3: start the Qwen3-8B encoder and the CLM server (first start downloads ~16 GB).
import json, time, requests
def tail(path, n=40): print(''.join(open(path).readlines()[-n:]))
def wait(url, proc, name, log, tries=180):
    for _ in range(tries):
        try:
            if requests.get(url, timeout=10).ok: print(name, 'is ready'); return
        except requests.RequestException: pass
        if proc.poll() is not None:
            tail(log); raise RuntimeError(f'{name} stopped; its log is above. Paste it to Claude.')
        time.sleep(10)
    tail(log); raise TimeoutError(f'{name} did not start; its log is above. Paste it to Claude.')
encoder = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3-8B', '--revision', 'b968826d9c46dd6066d109eabc6255188de91218',
                            '--served-model-name', 'qwen3-8b', '--runner', 'pooling', '--max-model-len', '2048',
                            '--dtype', 'bfloat16', '--port', '8090'], stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8090/health', encoder, 'Encoder (vLLM)', '/content/vllm.log')
server = subprocess.Popen(['clm-serve'], stdout=open('/content/clm_server.log', 'w'), stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8700/health', server, 'CLM server', '/content/clm_server.log')
probe = {'state': 'Bank of America arranged a 500 million dollar syndicated loan for a utility.',
         'questions': {'q1': {'type': 'noul', 'instructions': 'Does the article describe a bank lending money?'},
                       'q2': {'type': 'noul', 'instructions': 'Is the article about crude oil?',
                              'criteria': {'true': 'Oil prices or production.', 'false': 'Anything else.'}}}}
r = requests.post('http://127.0.0.1:8700/v1/systemone', json=probe, timeout=120)
print('test question:', r.status_code, json.dumps(r.json())[:600])
json.dump({'probe': r.json(), 'health': requests.get('http://127.0.0.1:8700/health').json()}, open(f'{SAVE}/logs/clm_probe.json', 'w'), indent=1)
assert r.ok, 'PROBLEM: paste this output to Claude'
MODEL_NAME = r.json().get('model')
OVER = ['--set', f'models.clm.expected_model_returned={MODEL_NAME}'] if MODEL_NAME else ['--set', 'models.clm.allow_returned_model_mismatch=true']

In [ ]:
# Step 4: run every dataset in priority order. Finished work is skipped; unfinished work resumes.
from google.colab import files
G = ['--set', 'models.clm.enabled=true', '--set', 'models.clm.quantization=none', '--set', 'models.clm.concurrency=32']
QUEUE = [  # (name, config, extra flags, needs HF token)
    ('reuters_v1',          'configs/native_reuters_top10_local.yaml',      ['--no-dev-eval'], False),
    ('reuters_v2_1',        'configs/native_reuters_top10_v2_1_local.yaml', ['--no-dev-eval'], False),
    ('reuters_v3_1',        'configs/native_reuters_top10_v3_1_local.yaml', ['--no-dev-eval'], False),
    ('fomc',                'configs/bench_fomc.yaml',              G + ['--no-dev-eval'], False),
    ('edgar',               'configs/bench_edgar.yaml',             G + ['--no-dev-eval'], False),
    ('credit_agreements',   'configs/bench_credit_agreements.yaml', G + ['--no-dev-eval'], True),
    ('wiki_industry',       'configs/bench_wiki_industry.yaml',     G + ['--no-dev-eval'], False),
    ('usnews',              'configs/bench_usnews.yaml',            G + ['--no-dev-eval'], False),
    ('tradetheevent',       'configs/bench_tradetheevent.yaml',     G + ['--no-dev-eval'], False),
    ('reuters_gsib',        'configs/bench_reuters.yaml',           G + ['--no-dev-eval'], False),
    ('fincausal',           'configs/fincausal.yaml',               G + ['--no-dev-eval'], False),
    ('bloomberg_full',      'configs/classification.yaml',          G, False),   # 445,391 articles; includes the dev-set check
]
import yaml
def corpus_ready(cfg):
    d = yaml.safe_load(open(cfg))['corpus']['directory']
    return os.path.exists(f'{d}/manifest.json')
def sh(cmd, log):
    with open(log, 'a') as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            fh.write(line)
            if any(k in line for k in ('articles,', 'session end', 'FATAL', 'ERROR', 'Traceback', 'macro', 'cost $', 'wall time')):
                print(line, end='')
        return p.wait()
for name, cfg, extra, gated in QUEUE:
    if os.path.exists(f'{SAVE}/logs/{name}.done'):
        print(f'== {name}: already done'); continue
    if gated and not os.environ.get('HF_TOKEN'):
        print(f'== {name}: skipped (no HF_TOKEN secret)'); continue
    log = f'{SAVE}/logs/{name}.log'
    print(f'\n== {name}')
    if not corpus_ready(cfg):
        print('   preparing the dataset (download + dedupe)...')
        if sh(['classify', 'prepare', '-c', cfg], log) != 0:
            print(f'   prepare failed: see {log}; continuing with the next dataset'); continue
    for attempt in range(3):  # a run can end with a few failed items; resume until the status is COMPLETE
        rc = sh(['classify', 'run', '-c', cfg, '-m', 'clm', '--resume'] + extra + OVER, log)
        ends = [l for l in open(log) if 'session end' in l]
        complete = rc == 0 and ends and 'status COMPLETE' in ends[-1]
        if complete or rc != 0:
            break
    if complete:
        open(f'{SAVE}/logs/{name}.done', 'w').write('ok')
        subprocess.run(f'cd {SAVE} && zip -qr /content/clm_{name}.zip logs/{name}.log reports $(grep -o "classification_results[_a-z]*/clm-[0-9a-f]*" logs/{name}.log | sort -u)', shell=True)
        files.download(f'/content/clm_{name}.zip')
    else:
        print(f'   run not complete (exit {rc}): see {log}. Run all again to resume.')
        break
print('\nAll done.' if all(os.path.exists(f'{SAVE}/logs/{n}.done') or (g and not os.environ.get('HF_TOKEN')) for n, _, _, g in QUEUE) else '\nNot finished yet: Run all again to continue.')